# Latent gate: Part C only, fine-tuning SIM-CoT-1B

Runs Part C of Amendment 1 in `results/latent_gate/heldout_scale/PLAN.md`. It is untriggered and exploratory. B3, the
re-anchoring runs, already finished in `DSA_Latent_Gate_Anchor_PartC_Colab.ipynb`; this notebook does not repeat them.

- **Training.** SIM-CoT-1B is fine-tuned three ways: `ft6`, `ft12` and `gate12_fixed`.
- **Scoring.** Each arm is scored on validation, and the plan's rule is applied.
  - **Download 1** follows. Send it as soon as it arrives.
- **Held-out sets.** An arm that passes the rule is scored on them, beside `ft6`.
  - **Download 2** follows, only if an arm passed.
- **Time.** About 3–4.5 hours. The smoke run near the start prints a training-time estimate.

**Before running:**
- **Runtime:** Runtime → Change runtime type → **A100 GPU**, then Runtime → **Run all**.
  - A full fine-tune of a 1B model does not fit on an L4 or T4: on an L4, even micro-batches of 4 ran out of memory.
  - The first cell stops at once on a GPU with less than 40 GB.
- **Drive.** Part B's records must already be on Drive, under `latent-gate/heldout-scale/`. This adds under 0.1 GB. The
  fine-tuned weights, about 5 GB per arm, stay on the Colab disk.
- **Resuming.** Validation scores resume from Drive. Trained weights don't survive a disconnect, so an arm whose scores are
  unfinished is retrained.
- **Keep the tab open.** If the browser asks whether this site may download multiple files, allow it.

In [ ]:
# Settings. Rerunning with the same settings resumes; finished pieces are skipped.
PART_C_MODEL = "simcot-llama1b"   # fixed by Amendment 1
REPO_REF = "main"                 # used only if B3's commit pin is missing on Drive
BATCH = 32                        # Part B's evaluation batch

import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > A100 GPU, then Runtime > Run all.")
name, memory = [part.strip() for part in gpu.stdout.splitlines()[0].split(",")]
print(f"GPU: {name}, {int(memory) / 1024:.0f} GiB")
if int(memory) < 38_000:
    raise SystemExit(f"Part C needs an A100 (40 GB or more); this runtime has {name}. "
                     "Runtime > Change runtime type > A100 GPU, then Runtime > Run all.")

In [ ]:
from pathlib import Path
import hashlib, json, os, runpy, shutil, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
OUT = DRIVE / "latent-gate" / "heldout-scale"

PINNED = OUT / "commit-amendment1.txt"  # written by the B3 run: Part C uses the same code
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
plan = Path("results/latent_gate/heldout_scale/PLAN.md")
recorded = Path("results/latent_gate/heldout_scale/PLAN.sha256").read_text().split()[0]
assert hashlib.sha256(plan.read_bytes()).hexdigest() == recorded, "PLAN.md does not match PLAN.sha256"
assert "## Amendment 1" in plan.read_text() and Path("scripts/latent_gate/partc_train.py").exists(), \
    f"{ref} predates Amendment 1"
PINNED.write_text(COMMIT)
print("Code:", COMMIT, f"(from {ref})")
free = shutil.disk_usage("/content").free / 2 ** 30
print(f"Colab disk: {free:.0f} GiB free")
if free < 60:
    print("Warning: training needs about 50 GiB of local disk (weights and a resume file); this may run out.")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = "/content/hf-cache"  # local disk: the 1B checkpoint stays off Drive
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation; results are unchanged
LOCAL = Path("/content/partc")  # fine-tuned 1B weights, on the Colab disk only
LOCAL.mkdir(exist_ok=True)
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=OUT / "logs")

run("scripts/setup_data.py", "--eval-only")
assert (DRIVE / "data/gsm_train.json").exists(), "Part C needs data/gsm_train.json on Drive (Stage 1's)"

SIZES = {"gsm8k_valid": 500, "gsm8k_test": 1319, "gsm_hard": 1319, "multiarith": 180, "svamp": 1000}
HELDOUT = ["gsm8k_test", "gsm_hard", "multiarith", "svamp"]

def finished(folder, dataset):
    path = folder / f"{dataset}.jsonl"
    return path.exists() and len(path.read_text().splitlines()) == SIZES[dataset]

def evaluate(name, source, datasets, policies, *extra):
    """Score `source` (["--llama", x, "--weights", dir]) into OUT/name, skipping finished datasets."""
    folder = OUT / name
    todo = [d for d in datasets if not finished(folder, d)]
    if not todo:
        print(f"{name}: {', '.join(datasets)} already on Drive")
        return
    run("scripts/latent_gate/heldout_eval.py", *source, "--datasets", *todo, "--output-dir", folder,
        "--policies", *policies, "--batch", BATCH, "--device", "cuda", *extra)

missing = [d for d in ["gsm8k_valid"] + HELDOUT if not finished(OUT / PART_C_MODEL, d)]
assert not missing, f"Part B's records for {PART_C_MODEL} are missing on Drive: {missing}"

from google.colab import files

def bundle(name, paths):
    """Zip OUT's `paths` (folders or files) with the amendment report and the commit, and download it."""
    target = Path("/content") / name
    shutil.rmtree(target, ignore_errors=True)
    target.mkdir()
    for path in paths:
        source = OUT / path
        if source.is_dir():
            shutil.copytree(source, target / path)
        elif source.exists():
            (target / path).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target / path)
    report = subprocess.run([PYTHON, "scripts/latent_gate/amendment_report.py", str(OUT)],
                            capture_output=True, text=True, cwd=REPO_DIR)
    (target / "amendment_report.txt").write_text(report.stdout + report.stderr)
    (target / "commit.txt").write_text(COMMIT)
    archive = shutil.make_archive(str(target), "zip", target)
    files.download(archive)
    print(f"Downloading {archive}. If it does not start, get it from the Files panel on the left.")

## Part C: training and validation

1. **A two-update smoke run** of `gate12_fixed` checks memory. It uses micro-batches of 8, or 4 if 8 does not fit, and
   prints a time estimate.
2. **Training and scoring.** `ft6`, `ft12` and `gate12_fixed` are each trained (one pass, Stage 1's recipe) and scored on
   validation.
3. **The rule.** The report applies the plan's rule, and Download 1 follows.

In [ ]:
ARMS = {"ft6": ["plain"], "ft12": ["plain"], "gate12_fixed": ["plain", "gate", "gate_soft"]}
HELDOUT_POLICY = {"ft6": "plain", "ft12": "plain", "gate12_fixed": "gate"}
MICRO = None

def weights(arm):
    """The fine-tuned model directory on the Colab disk, training the arm first if it is not there."""
    global MICRO
    folder = LOCAL / arm
    if (folder / "model/pytorch_model.bin").exists():
        return folder / "model"
    if MICRO is None:
        for micro in (8, 4):
            try:
                run("scripts/latent_gate/partc_train.py", "--llama", PART_C_MODEL, "--arm", "gate12_fixed",
                    "--output", LOCAL / "smoke", "--micro-batch", micro, "--device", "cuda", "--bf16", "--smoke")
                MICRO = micro
                break
            except Exception as error:
                print(f"micro-batch {micro} did not run: {error}")
        assert MICRO, "Part C does not fit on this GPU even with micro-batches of 4"
    run("scripts/latent_gate/partc_train.py", "--llama", PART_C_MODEL, "--arm", arm, "--output", folder,
        "--micro-batch", MICRO, "--device", "cuda", "--bf16")
    logs = OUT / "arms" / f"{PART_C_MODEL}-{arm}"
    logs.mkdir(parents=True, exist_ok=True)
    for name in ("args.json", "metrics.jsonl"):
        shutil.copy2(folder / name, logs / name)
    if (folder / "model/gate.pt").exists():
        shutil.copy2(folder / "model/gate.pt", logs / "gate.pt")
    return folder / "model"

for arm, policies in ARMS.items():
    name = f"{PART_C_MODEL}-{arm}"
    if not finished(OUT / name, "gsm8k_valid"):
        evaluate(name, ["--llama", PART_C_MODEL, "--weights", weights(arm)], ["gsm8k_valid"], policies)
verdict = OUT / "partc_verdict.json"
run("scripts/latent_gate/amendment_report.py", OUT, "--verdict", verdict)
passing = json.loads(verdict.read_text())["passing"]
print("Arms passing the Part C rule:", passing or "none")

# Download 1: validation scores, training logs, gates and the verdict. Send it as soon as it arrives.
PARTC = [f"{PART_C_MODEL}-{arm}" for arm in ARMS] + [f"arms/{PART_C_MODEL}-{arm}" for arm in ARMS]
bundle("latent-gate-partc-valid", PARTC + ["partc_verdict.json"])  # not arms/ itself: Part A's GPT-2 weights

## Held-out confirmation

This runs only if an arm passed. Each passing arm is scored on the held-out sets beside `ft6`, and Download 2 follows.

In [ ]:
if passing:
    for arm in ["ft6"] + passing:
        evaluate(f"{PART_C_MODEL}-{arm}", ["--llama", PART_C_MODEL, "--weights", weights(arm)], HELDOUT,
                 [HELDOUT_POLICY[arm]])
    run("scripts/latent_gate/amendment_report.py", OUT)
    # Download 2: everything from Part C, with the held-out scores.
    bundle("latent-gate-partc", PARTC + ["partc_verdict.json"])
else:
    print("No arm passed, so nothing runs on the held-out sets. Download 1 holds all of Part C.")

**Reading the results**
- **The rule.** An arm passes if, at 12 steps, it beats `ft6`@6 on problems with 4+ steps at p < 0.025. It must also not
  be lower overall, and must not drop from 12 to 24 steps at p < 0.05.
- **The confirmation.** A gain counts only if, on the held-out primary pool, the arm at 12 steps beats both `ft6`@6 and
  the base model's `plain`@6 at p < 0.05.

**Sharing:** send Download 1 as soon as it arrives, and Download 2 if it appears.